# Optional: train a football-specific YOLO model on Google Colab

The app works with the COCO-pretrained `yolo26n.pt`, which only knows `person` and `sports ball`.
A model fine-tuned on football images can separate **player**, **goalkeeper**, **referee** and **ball**.
This notebook trains one on a free Colab GPU.

**Before you start**
1. In Colab choose *Runtime > Change runtime type > T4 GPU*.
2. Create a free account at roboflow.com and copy your private API key (Settings > API keys).
3. Open a football dataset on Roboflow Universe (search for "football players detection"), and note the
   workspace name, project name and version number shown in its download dialog. Check the dataset licence.

Training for 50 epochs at 1280 px takes roughly one to two hours on a T4 GPU.

In [ ]:
!pip install -q ultralytics==8.4.177 roboflow
import ultralytics
ultralytics.checks()

## 1. Download the dataset in YOLO format

In [ ]:
from getpass import getpass
from roboflow import Roboflow

WORKSPACE = input("Roboflow workspace: ")
PROJECT = input("Roboflow project: ")
VERSION = int(input("Dataset version number: "))

rf = Roboflow(api_key=getpass("Roboflow API key: "))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
print("Dataset saved in", dataset.location)
print(open(f"{dataset.location}/data.yaml").read())

## 2. Train

Small players and the ball need a large input size, so `imgsz=1280` is used. Lower `batch` if the GPU runs out
of memory.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")  # start from COCO weights (transfer learning)
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=1280,
    batch=8,
    patience=15,
    device=0,
    project="football",
    name="yolo26n_football",
)

## 3. Evaluate on the validation set

In [ ]:
best = YOLO("football/yolo26n_football/weights/best.pt")
metrics = best.val(data=f"{dataset.location}/data.yaml", imgsz=1280)
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("Class names:", best.names)

## 4. Use the model in the app

Download `best.pt`, rename it to `football.pt` and copy it into the project's `models/` folder.
It then appears as "Custom football model" in the model list on the Analyse page. The class names are read
from the file, so referees and goalkeepers are reported separately.

In [ ]:
from google.colab import files
import shutil
shutil.copy("football/yolo26n_football/weights/best.pt", "football.pt")
files.download("football.pt")